# Fit your own behavioral data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-hahn/bayesian-models-of-perception/blob/main/code/Demo/Fit_Your_Own_Data.ipynb)

This notebook validates a CSV and fits the Bayesian perception model without requiring command-line work. It runs both in Google Colab and in a local Jupyter installation.

A fit can take a while. Start with one loss exponent (`P_VALUES = [2]`) and one fold. The small example CSVs in this repository demonstrate the format only; they are not large enough for meaningful inference.

## 1. Set up the demo

In Colab, this clones the repository. In either environment, it locates the demo and installs the pinned dependencies only if a required package is missing.

In [ ]:
from importlib.util import find_spec
from pathlib import Path
import subprocess
import sys

IN_COLAB = find_spec("google.colab") is not None

if IN_COLAB:
    REPO_DIR = Path("/content/bayesian-models-of-perception")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/m-hahn/bayesian-models-of-perception.git", str(REPO_DIR)],
            check=True,
        )
    DEMO_DIR = REPO_DIR / "code" / "Demo"
else:
    candidates = [Path.cwd(), Path.cwd() / "code" / "Demo", Path.cwd() / "Demo"]
    DEMO_DIR = next(
        (candidate.resolve() for candidate in candidates
         if (candidate / "run_behavioral_pipeline.py").exists()),
        None,
    )
    if DEMO_DIR is None:
        raise RuntimeError("Run this notebook from the repository or code/Demo directory.")

required = ["matplotlib", "numpy", "scipy", "torch"]
if any(find_spec(package) is None for package in required):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(DEMO_DIR.parent / "requirements.txt")],
        check=True,
    )

sys.path.insert(0, str(DEMO_DIR))
from run_behavioral_pipeline import run_pipeline, validate_csv

print(f"Demo directory: {DEMO_DIR}")
print(f"Python: {sys.executable}")

## 2. Upload a CSV

The CSV must have a header and the columns shown below. `condition` is optional; when omitted, the model uses condition 5 for circular data or condition 4 for interval data.

```csv
condition,stimulus,response
5,110.0,87.0
5,276.0,262.6
```

Circular values must be in `[0, 360)`. For orientations measured modulo 180 degrees, transform both stimulus and response with `(2 * value) % 360` before fitting. Interval values must be in `[0, 3]`.

In [ ]:
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
else:
    try:
        from ipywidgets import FileUpload
        from IPython.display import display
        uploader = FileUpload(accept=".csv,text/csv", multiple=False)
        display(uploader)
        print("Choose a CSV above, then run the next cell.")
    except ImportError:
        print("ipywidgets is unavailable. Set INPUT_CSV to a local CSV path in the next cell.")

In [ ]:
UPLOAD_DIR = DEMO_DIR / "input" / "uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if not uploaded:
        raise RuntimeError("Upload a CSV in the previous cell first.")
    upload_name, upload_content = next(iter(uploaded.items()))
elif "uploader" in globals():
    if not uploader.value:
        raise RuntimeError("Choose a CSV in the previous cell first.")
    if isinstance(uploader.value, dict):  # ipywidgets 7
        upload_name, upload_item = next(iter(uploader.value.items()))
    else:  # ipywidgets 8
        upload_item = uploader.value[0]
        upload_name = upload_item["name"]
    upload_content = upload_item["content"]
else:
    # Edit this fallback when widgets are unavailable.
    INPUT_CSV = DEMO_DIR / "input" / "example_circular.csv"
    upload_name = upload_content = None

if upload_name is not None:
    INPUT_CSV = UPLOAD_DIR / Path(upload_name).name
    INPUT_CSV.write_bytes(bytes(upload_content))

print(f"Using: {INPUT_CSV}")
print(INPUT_CSV.read_text().splitlines()[:6])

## 3. Configure and validate

Choose `SPACE = "circular"` or `SPACE = "interval"`. Validation checks the schema, numeric values, condition IDs, and stimulus-space bounds without starting a fit or writing model inputs. Warnings about incomplete coverage should be considered before interpreting fitted priors.

In [ ]:
SPACE = "circular"       # "circular" or "interval"
P_VALUES = [2]           # Start with one fit; later try 0, 1, 2, 4, 6, 8
FOLD = 0
REG_WEIGHT = 10.0
GRID = None              # Uses 180 for circular or 400 for interval
WRAP_CIRCULAR = False    # Set True only if modulo-360 wrapping is intended
OVERWRITE = False        # Set True to rerun exactly the same configuration
PLOT_EVERY = 1000        # Set 0 to disable circular diagnostic figures

summary = validate_csv(
    INPUT_CSV,
    SPACE,
    wrap_circular=WRAP_CIRCULAR,
)
print(f"Validated {summary.row_count} rows")
print(f"Conditions: {summary.conditions}")
print(f"Stimulus range: {summary.stimulus_range}")
print(f"Response range: {summary.response_range}")

## 4. Run the fit

This cell is the long-running step. Its detailed optimization output is expected. A CUDA device is selected automatically when PyTorch can use one. Interrupt the cell to cancel the active fit.

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {DEVICE}")

result = run_pipeline(
    INPUT_CSV,
    SPACE,
    p=P_VALUES,
    fold=FOLD,
    reg_weight=REG_WEIGHT,
    grid=GRID,
    wrap_circular=WRAP_CIRCULAR,
    overwrite=OVERWRITE,
    device=DEVICE,
    plot_every=PLOT_EVERY,
)

## 5. Review and download results

The loss file begins with the held-out cross-validation loss. Lower values are better when comparing fits evaluated on the same observations and fold. The parameter log contains the fitted model state; circular fits may also include a PDF diagnostic figure.

In [ ]:
from IPython.display import FileLink, display
from zipfile import ZIP_DEFLATED, ZipFile

for fit in result.fits:
    print(f"p={fit.p}: cross-validation loss = {fit.cross_validation_loss:.6g}")
    display(FileLink(str(fit.loss_path)))
    display(FileLink(str(fit.parameter_log_path)))
    if fit.figure_path is not None and fit.figure_path.exists():
        display(FileLink(str(fit.figure_path)))

RESULTS_ZIP = DEMO_DIR / f"{INPUT_CSV.stem}-{SPACE}-fit-results.zip"
with ZipFile(RESULTS_ZIP, "w", ZIP_DEFLATED) as archive:
    archive.write(INPUT_CSV, f"input/{INPUT_CSV.name}")
    archive.write(result.legacy_dataset_path, f"converted/{result.legacy_dataset_path.name}")
    for fit in result.fits:
        for path in (fit.loss_path, fit.parameter_log_path, fit.figure_path):
            if path is not None and path.exists():
                archive.write(path, f"results/{path.name}")

print(f"Bundled results: {RESULTS_ZIP}")
if IN_COLAB:
    files.download(str(RESULTS_ZIP))
else:
    display(FileLink(str(RESULTS_ZIP)))